# Training a Gravitational-Wave Detector with Sage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnarenraju/sage/blob/main/notebooks/colab/03_training_and_evaluation.ipynb)

This notebook walks through building, training, and evaluating a neural network for GW detection using Sage's `MSCNN1D_2DResNetCBAM` architecture.

**Structure:**
- **Part A** — Architecture overview and forward pass
- **Part B** — Train from scratch on synthetic data (~10 min on T4)
- **Part C** — Load a pretrained checkpoint and evaluate with a ROC curve

**Runtime:** ~25 min on T4

## Setup

In [ ]:
import subprocess, sys
try:
    import sage
    print('Sage already installed.')
except ImportError:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/nnarenraju/sage.git'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'sage/'], check=True)
    print('Sage installed.')


In [ ]:
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_curve, auc

from pycbc.psd import aLIGOZeroDetHighPower
from sage.core.base_classes import BaseConfig, BaseDataConfig
from sage.core.config import register_configs
from sage.data.waveform.approximants.IMRPhenomD import IMRPhenomD
from sage.data.waveform.project import ConstantProjection
from sage.architecture.network import MSCNN1D_2DResNetCBAM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
class TutorialCFG:
    batch_size    = 64
    device        = device
    dtype         = torch.float32
    detectors     = ['H1', 'L1']
    do_point_estimate = []   # detection only — no parameter regression
    class_balance = 0.5
    clip_norm     = 1.0
    autocast      = False

class TutorialDataCFG:
    sample_rate                 = 2048.0
    signal_low_frequency_cutoff = 20.0
    sample_length_in_s          = 8.0
    padding_length_in_s         = 2.0

register_configs(BaseConfig(TutorialCFG()), BaseDataConfig(TutorialDataCFG()))
print('Configs registered.')


---
## Part A — Architecture

`MSCNN1D_2DResNetCBAM` (Multi-Scale CNN + 2D ResNet + Convolutional Block Attention Module) is Sage's primary detection backbone.

```
Input (B, D=2, T)
      │
  InstanceNorm1d
      │
  Multi-scale 1D CNN frontend (per detector)
  ├─ Conv1D(kernel=16)   ┐
  ├─ Conv1D(kernel=32)   ├─ concatenate → feature map (B, D×scales, T')
  └─ Conv1D(kernel=64)   ┘
      │
  2D ResNet-50 + CBAM attention backend
      │
  AdaptiveAvgPool1d(512) → Flatten
      │
  Linear(512 → 1)  →  ranking statistic (detection logit)
```


In [ ]:
model = MSCNN1D_2DResNetCBAM(
    frontend_filters     = 16,   # smaller than production (32) for tutorial speed
    frontend_kernel      = 64,
    backend_resnet_size  = 18,   # ResNet-18 instead of 50 for tutorial speed
    norm_type            = 'instancenorm',
).to(device=device, dtype=torch.float32)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')


In [ ]:
# Verify the forward pass on a random batch
seg_len = int(TutorialDataCFG.sample_rate * TutorialDataCFG.sample_length_in_s)
dummy   = torch.randn(4, 2, seg_len, device=device, dtype=torch.float32)

with torch.no_grad():
    logits, point_ests = model(dummy)

print(f'Input shape:            {list(dummy.shape)}')
print(f'Detection logit shape:  {list(logits.shape)}')
print(f'Point estimates shape:  {list(point_ests.shape)}  (empty — no regression in tutorial)')


---
## Part B — Training from Scratch

We build a simple on-the-fly data generator:
- **Signal samples**: IMRPhenomD waveform → project onto H1/L1 → whiten with aLIGO design ASD → IFFT
- **Noise samples**: Gaussian white noise (flat after whitening by construction)

The training batch mixes signals and noise, and we minimise binary cross-entropy on the detection logit.

Training 5 epochs × 200 steps at batch size 64 takes roughly **10 minutes on a T4 GPU**.

In [ ]:
# Frequency grid
f_l, f_u, del_f = 20.0, 1024.0, 1.0 / 8.0  # df = 1/8 Hz matches 8-s segment
n_waveform = int(round((f_u - f_l) / del_f)) + 1
n_padded   = int(round(f_u / del_f)) + 1
seg_len    = int(TutorialDataCFG.sample_rate * TutorialDataCFG.sample_length_in_s)

# Design ASD (for whitening)
psd_np  = np.array(aLIGOZeroDetHighPower(n_padded, del_f, f_l).data[:])
psd_np[:int(f_l / del_f)] = 1.0  # avoid /0; signal is zero here anyway
asd_dev = torch.tensor(np.sqrt(psd_np), dtype=torch.float64, device=device)  # (n_padded,)

pwave = ConstantProjection()
rng = np.random.default_rng(42)

print(f'Segment length:    {seg_len} samples ({seg_len/2048:.1f} s)')
print(f'FD padded bins:    {n_padded}')


In [ ]:
def make_batch(batch_size, snr_range=(6., 20.)):
    """Build a mixed batch of whitened signal + noise samples.
    Returns (x, labels) where x has shape (B, 2, seg_len) float32.
    """
    half = batch_size // 2

    # --- Signal generation ---
    m1 = torch.tensor(rng.uniform(7., 50., half), dtype=torch.float64, device=device)
    m2 = torch.tensor(rng.uniform(7., 50., half), dtype=torch.float64, device=device)
    m1, m2 = torch.maximum(m1, m2), torch.minimum(m1, m2)
    chi1 = torch.tensor(rng.uniform(-0.99, 0.99, half), dtype=torch.float64, device=device)
    chi2 = torch.tensor(rng.uniform(-0.99, 0.99, half), dtype=torch.float64, device=device)
    dist = torch.tensor(rng.uniform(100., 1000., half), dtype=torch.float64, device=device)
    tc   = torch.zeros(half, dtype=torch.float64, device=device)
    phic = torch.tensor(rng.uniform(0., 2*np.pi, half), dtype=torch.float64, device=device)
    incl = torch.tensor(np.arccos(rng.uniform(-1., 1., half)), dtype=torch.float64, device=device)
    pol  = torch.tensor(rng.uniform(0., np.pi, half), dtype=torch.float64, device=device)
    ra   = torch.tensor(rng.uniform(0., 2*np.pi, half), dtype=torch.float64, device=device)
    dec  = torch.tensor(np.arcsin(rng.uniform(-1., 1., half)), dtype=torch.float64, device=device)

    params = torch.stack([m1, m2, chi1, chi2, dist, tc, phic, incl, pol, ra, dec], dim=1)

    f = (f_l + del_f * torch.arange(n_waveform, dtype=torch.float64, device=device)) \
        .unsqueeze(0).expand(half, -1).clone()
    f_ref = torch.full((half, 1), f_l, dtype=torch.float64, device=device)

    hp, hc = IMRPhenomD(f, f_ref)(params, reproduce_lal=True)
    # hp, hc: (half, n_padded) complex

    signal_fd = pwave(hp, hc, ra=ra, dec=dec, polarization=pol)  # (half, 2, n_padded)

    # Whiten and IFFT
    signal_fd_white = signal_fd / asd_dev               # (half, 2, n_padded)
    signal_td = torch.fft.irfft(signal_fd_white, n=2*(n_padded-1))  # (half, 2, T_full)
    # Crop to seg_len (last seg_len samples contain the merger)
    signal_td = signal_td[:, :, -seg_len:].real.float()  # (half, 2, seg_len)

    # --- Noise generation (white — already whitened by construction) ---
    noise_td = torch.randn(half, 2, seg_len, device=device, dtype=torch.float32)

    # Combine and shuffle
    x = torch.cat([signal_td, noise_td], dim=0)             # (B, 2, seg_len)
    labels = torch.cat([
        torch.ones(half, device=device),
        torch.zeros(half, device=device)
    ])
    perm = torch.randperm(batch_size, device=device)
    return x[perm], labels[perm]

# Quick test
x_test, y_test = make_batch(8)
print(f'Batch shape: {x_test.shape}, labels: {y_test}')


In [ ]:
# --- Optimiser and scheduler ---
optimiser = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=200 * 5, eta_min=1e-6
)

BATCH_SIZE   = 64
NUM_ITERS    = 200
NUM_EPOCHS   = 5

train_losses = []

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for _ in tqdm(range(NUM_ITERS), desc=f'Epoch {epoch+1}/{NUM_EPOCHS}'):
        x, labels = make_batch(BATCH_SIZE)

        optimiser.zero_grad(set_to_none=True)
        logits, _ = model(x)              # (B, 1)
        loss = F.binary_cross_entropy_with_logits(
            logits.squeeze(-1), labels
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        scheduler.step()

        epoch_loss += loss.item()

    avg = epoch_loss / NUM_ITERS
    train_losses.append(avg)
    print(f'  loss = {avg:.4f}')

print('Training complete.')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, NUM_EPOCHS + 1), train_losses, 'o-')
ax.axhline(np.log(2), color='k', ls='--', label='random-guess BCE = ln(2)')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.set_title('Training loss (mini-trained model)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Quick accuracy check on held-out synthetic test batch

In [ ]:
model.eval()
with torch.no_grad():
    x_val, y_val = make_batch(512)
    logits_val, _ = model(x_val)
    preds = (logits_val.squeeze(-1) > 0).float()
    acc   = (preds == y_val).float().mean().item()

print(f'Mini-trained model accuracy (512 samples): {100*acc:.1f}%')
print('(random chance = 50%; well-trained production model achieves >90%)')


---
## Part C — Pretrained Checkpoint Evaluation

A model trained from scratch for 5 epochs on synthetic data is not production-quality. The pretrained checkpoint below was trained for many hours on real O3a noise with the full Sage pipeline.

If the checkpoint URL is not yet available, this section is skipped gracefully — but it demonstrates the evaluation workflow.

In [ ]:
import urllib.request, os

CHECKPOINT_URL = (
    'https://github.com/nnarenraju/sage/releases/download/tutorial-v1/'
    'sage-tutorial-checkpoint.pt'
)
CHECKPOINT_PATH = 'sage-tutorial-checkpoint.pt'

checkpoint_available = False
try:
    if not os.path.exists(CHECKPOINT_PATH):
        print('Downloading pretrained checkpoint ...')
        urllib.request.urlretrieve(CHECKPOINT_URL, CHECKPOINT_PATH)
    checkpoint_available = True
    print('Checkpoint ready.')
except Exception as e:
    print(f'Checkpoint not yet available ({e}).')
    print('Skipping Part C — run Part B results are shown instead.')


In [ ]:
if checkpoint_available:
    pretrained_model = MSCNN1D_2DResNetCBAM(
        frontend_filters    = 32,
        frontend_kernel     = 64,
        backend_resnet_size = 50,
        norm_type           = 'instancenorm',
    ).to(device=device, dtype=torch.float32)

    state = torch.load(CHECKPOINT_PATH, map_location=device)
    pretrained_model.load_state_dict(state)
    pretrained_model.eval()
    print(f'Pretrained model loaded ({sum(p.numel() for p in pretrained_model.parameters()):,} params)')
else:
    pretrained_model = model  # fall back to mini-trained
    print('Using mini-trained model for evaluation.')


In [ ]:
# Generate a larger synthetic test set: 1000 signal + 1000 noise
N_TEST = 2000
all_logits = []
all_labels = []

pretrained_model.eval()
with torch.no_grad():
    for _ in tqdm(range(N_TEST // 64 + 1), desc='Evaluating'):
        x_b, y_b = make_batch(64)
        l_b, _ = pretrained_model(x_b)
        all_logits.append(l_b.squeeze(-1).cpu())
        all_labels.append(y_b.cpu())

all_logits = torch.cat(all_logits)[:N_TEST]
all_labels = torch.cat(all_labels)[:N_TEST]
scores = torch.sigmoid(all_logits).numpy()
labels_np = all_labels.numpy()
print(f'Evaluated {N_TEST} samples.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score histogram
axes[0].hist(scores[labels_np == 1], bins=40, alpha=0.7, label='Signal', color='C0')
axes[0].hist(scores[labels_np == 0], bins=40, alpha=0.7, label='Noise',  color='C1')
axes[0].set_xlabel('Detection score  σ(logit)')
axes[0].set_ylabel('Count')
axes[0].set_title('Score distribution: signal vs. noise')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ROC curve
fpr, tpr, _ = roc_curve(labels_np, scores)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate')
axes[1].set_title('ROC curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f'AUC = {roc_auc:.4f}  (1.0 = perfect, 0.5 = random)')


## Summary

| | Mini-trained (Part B) | Pretrained (Part C) |
|---|---|---|
| Architecture | ResNet-18, 16 filters | ResNet-50, 32 filters |
| Training | 5 epochs, synthetic noise | Full pipeline, real O3a noise |
| Expected AUC | ~0.7–0.8 | ~0.95+ |

To train a production-quality model, see the [User Guide](https://sage-gw.readthedocs.io/en/latest/) and the run scripts under [`runs/`](https://github.com/nnarenraju/sage/tree/main/runs).
